# V3 Risk Mapping Visualization

This notebook visualizes the expected risk scores generated by the V3 Modeling Pipeline (FCM + AutoML).
It uses PyDeck to create an interactive heat-map style projection of the Bangkok road network.

In [ ]:
import pandas as pd
import geopandas as gpd
import pydeck as pdk
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

### 1. Load Risk Data and Geometries

In [ ]:
DATA_DIR = "../../data/processed"

scores_path = os.path.join(DATA_DIR, "results/risk_scores_v3.parquet")
segments_path = os.path.join(DATA_DIR, "road_segments/road_segments.gpkg")

print(f"Loading risk scores from: {scores_path}")
scores = pd.read_parquet(scores_path)

print(f"Loading road geometries from: {segments_path}")
segments = gpd.read_file(segments_path, columns=['segment_id', 'geometry'])

# Merge Data
gdf = segments.merge(scores, on="segment_id", how="inner")
gdf = gdf.to_crs("EPSG:4326")

print(f"Total segments matched: {len(gdf):,}")
gdf.head()

### 2. Risk Distribution Analysis

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(gdf['risk_score'], bins=50, kde=True)
plt.title('Distribution of V3 Expected Risk Scores')
plt.xlabel('Risk Score (0-1)')
plt.ylabel('Number of Segments')
plt.show()

### 3. Interactive PyDeck Visualization

We convert the geometries to coordinate lists for PyDeck and define a continuous Yellow-to-Red color gradient based on the `risk_score`.

In [ ]:
# Filter out very low risk segments to improve map performance if needed
threshold = 0.15
gdf_map = gdf[gdf['risk_score'] >= threshold].copy()
print(f"Mapping {len(gdf_map):,} segments with risk >= {threshold*100}%")

# Convert Linestring to coordinate lists for PyDeck PathLayer
gdf_map['path'] = gdf_map['geometry'].apply(lambda geom: [[c[0], c[1]] for c in geom.coords])

def get_color(risk):
    """Continuous Heatmap: Yellow (low risk) to Red (high risk)"""
    r = int(255)
    g = int(255 * (1 - risk))
    b = int(0)
    a = int(100 + 155 * risk) # More transparent for lower risk
    return [r, g, b, a]

gdf_map['color'] = gdf_map['risk_score'].apply(get_color)

# Drop heavy geometry column as PyDeck uses 'path'
df_map = gdf_map.drop(columns=['geometry'])

layer = pdk.Layer(
    "PathLayer",
    df_map,
    pickable=True,
    get_color="color",
    width_scale=25,
    width_min_pixels=3,
    get_path="path",
    get_width=6,
)

view_state = pdk.ViewState(
    latitude=13.7563,
    longitude=100.5018,
    zoom=11,
    pitch=45
)

r = pdk.Deck(
    layers=[layer], 
    initial_view_state=view_state, 
    tooltip={"text": "Segment ID: {segment_id}\nV3 Risk: {risk_pct}%"}
)

r.show()